In [ ]:
import os

# Working directory must contain AlphaSimPy.py for imports
os.chdir(r"/Users/mtwatson/Library/CloudStorage/Box-Box/Projects/AI agent for breeding/Endpoint 2 agent")


# Martin Plant Breeding Program - AlphaSimPy Notebook

This notebook converts a BRAID breeding program abstraction into a tutorial-style AlphaSimPy workflow.

**Program**: Martin Plant Breeding Program  
**Horizon**: 5 years  
**Package**: AlphaSimPy

The source BRAID describes a five-stage plant breeding pipeline with:
- a current founder population,
- biparental crossing,
- phenotypic selection in Year 1,
- genomic/EBV-oriented selection in later years,
- test-cross style evaluation stages,
- and a final release decision.

This notebook follows the structure of the AlphaSimPy tutorials and implements a practical executable approximation of the BRAID design.


## Assumptions and Mapping from BRAID to AlphaSimPy

The BRAID abstraction contains several placeholders and diagram-derived assumptions. To make the program executable in AlphaSimPy, this notebook uses the following explicit assumptions:

1. **Genome simulation**
   - Diploid species with 10 chromosomes.
   - One additive trait named `overall_breeding_value`.
   - 100 QTL are modeled across the genome.
   - Founder haplotypes are generated with `runMacs`.

2. **Selection logic**
   - Year 1 uses **phenotypic selection**.
   - Year 2 and later use a simplified **EBV-style approximation** based on true genetic values plus noise, because the BRAID file specifies genomic selection conceptually but does not provide a full training population or marker model specification.
   - This keeps the notebook deterministic and runnable while preserving the intended stage progression.

3. **Crossing and advancement**
   - Biparental crossing is represented with `randCross`.
   - Stage sizes are matched as closely as possible to the BRAID population sizes.
   - Test-cross stages (`TC1`, `TC2`, `TC3`) are represented as **evaluation proxies** using replicated phenotypes and summary statistics rather than a separate heterotic tester design, because the BRAID abstraction does not define tester pools.

4. **Release decision**
   - The final released variety is represented by the top Year 5 selected individual(s) based on the final evaluation metric.

These assumptions are documented so the notebook remains transparent and easy to adapt.


## Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from AlphaSimPy import runMacs, SimParam, newPop, randCross, setPheno, selectInd, meanG, varG

print("AlphaSimPy BRAID Conversion Notebook")
print("Libraries imported successfully!")

## Global Parameters

In [ ]:
# Core program parameters extracted from BRAID
program_name = "Martin Plant Breeding Program"
n_years = 5
founder_size = 100
n_chr = 10
n_qtl = 100
h2 = 0.3

# Population sizes from BRAID
sizes = {
    "current_population": 100,
    "year1_candidates": 100,
    "year1_selected": 80,
    "gen1_progeny": 50,
    "year2_candidates": 80,
    "year2_selected": 60,
    "gen2_progeny": 40,
    "year3_candidates": 60,
    "year3_selected": 40,
    "gen3_progeny": 30,
    "year4_candidates": 40,
    "year4_selected": 20,
    "gen4_progeny": 20,
    "year5_candidates": 20,
    "year5_selected": 10,
    "gen5_progeny": 10
}

# Additional simulation settings
seg_sites_per_chr = 200
inbred_founders = False
random_seed = 12345

np.random.seed(random_seed)

print("Program:", program_name)
print("Years:", n_years)
print("Founder size:", founder_size)
print("Chromosomes:", n_chr)
print("QTL:", n_qtl)
print("Trait heritability:", h2)
print("Stage sizes:")
for k, v in sizes.items():
    print(f"  {k}: {v}")

## Helper Functions

In [ ]:
def summarize_stage(stage_name, pop):
    """Return a one-row summary dictionary for a population stage."""
    return {
        "stage": stage_name,
        "nInd": getattr(pop, "nInd", np.nan),
        "meanG": float(meanG(pop)),
        "varG": float(varG(pop))
    }

def add_noisy_ebv(pop, accuracy=0.6):
    """Create a simple EBV proxy from true genetic values plus Gaussian noise."""
    gv = np.array(pop.gv).reshape(-1)
    gv_sd = np.std(gv) if np.std(gv) > 0 else 1.0
    noise_sd = gv_sd * max(1e-6, (1.0 - accuracy) / max(accuracy, 1e-6))
    ebv = gv + np.random.normal(0, noise_sd, size=len(gv))
    return ebv

def select_top_by_scores(pop, scores, n_select):
    """Select top individuals using score ranking and AlphaSimPy selectInd fallback by phenotype order proxy."""
    order = np.argsort(scores)[::-1]
    top_idx = order[:n_select]
    # Use phenotype proxy so selectInd can operate on the desired ranking
    pop.pheno = np.array(scores).reshape(-1, 1)
    selected = selectInd(pop, nInd=n_select, use="pheno", trait=0, selectTop=True)
    return selected, top_idx

stage_records = []

## Simulate Founder Haplotypes and Create the Base Population

In [ ]:
# Simulate founder haplotypes
founder_genomes = runMacs(
    nInd=founder_size,
    nChr=n_chr,
    segSites=seg_sites_per_chr,
    inbred=inbred_founders
)

# Set simulation parameters
SP = SimParam(founder_genomes)
SP.addTraitA(nQtlPerChr=max(1, int(np.ceil(n_qtl / n_chr))))
SP.setVarE(h2=h2)

# Create the current population
current_population = newPop(founder_genomes, simParam=SP)

print("Founder population created.")
print("Current population mean genetic value:", meanG(current_population))
print("Current population genetic variance:", varG(current_population))

stage_records.append(summarize_stage("current_population", current_population))

## Five-Year Breeding Pipeline

In [ ]:
# -------------------------
# Year 1: phenotypic selection
# -------------------------
year1_candidates = current_population
setPheno(year1_candidates, h2=h2, simParam=SP)
year1_selected = selectInd(
    year1_candidates,
    nInd=sizes["year1_selected"],
    use="pheno",
    trait=0,
    selectTop=True
)
stage_records.append(summarize_stage("year1_candidates", year1_candidates))
stage_records.append(summarize_stage("year1_selected", year1_selected))

# Produce Gen1 progeny
gen1_progeny = randCross(
    year1_selected,
    nCrosses=sizes["gen1_progeny"],
    nProgeny=1,
    simParam=SP
)
stage_records.append(summarize_stage("gen1_progeny", gen1_progeny))

# -------------------------
# Year 2: genomic/EBV-style selection proxy
# -------------------------
year2_candidates = gen1_progeny
setPheno(year2_candidates, h2=h2, simParam=SP)
year2_ebv = add_noisy_ebv(year2_candidates, accuracy=0.60)
year2_selected, year2_idx = select_top_by_scores(
    year2_candidates,
    year2_ebv,
    sizes["year2_selected"]
)
stage_records.append(summarize_stage("year2_candidates", year2_candidates))
stage_records.append(summarize_stage("year2_selected", year2_selected))

gen2_progeny = randCross(
    year2_selected,
    nCrosses=sizes["gen2_progeny"],
    nProgeny=1,
    simParam=SP
)
stage_records.append(summarize_stage("gen2_progeny", gen2_progeny))

# TC1 proxy evaluation
setPheno(year2_candidates, h2=0.20, simParam=SP)
tc1_mean = float(np.mean(year2_candidates.pheno))
tc1_locations = 2

# -------------------------
# Year 3: continued advancement and GCA-oriented proxy
# -------------------------
year3_candidates = gen2_progeny
setPheno(year3_candidates, h2=h2, simParam=SP)
year3_ebv = add_noisy_ebv(year3_candidates, accuracy=0.65)
year3_selected, year3_idx = select_top_by_scores(
    year3_candidates,
    year3_ebv,
    sizes["year3_selected"]
)
stage_records.append(summarize_stage("year3_candidates", year3_candidates))
stage_records.append(summarize_stage("year3_selected", year3_selected))

gen3_progeny = randCross(
    year3_selected,
    nCrosses=sizes["gen3_progeny"],
    nProgeny=1,
    simParam=SP
)
stage_records.append(summarize_stage("gen3_progeny", gen3_progeny))

# -------------------------
# Year 4: intermediate testing
# -------------------------
year4_candidates = gen3_progeny
setPheno(year4_candidates, h2=h2, simParam=SP)
year4_ebv = add_noisy_ebv(year4_candidates, accuracy=0.70)
year4_selected, year4_idx = select_top_by_scores(
    year4_candidates,
    year4_ebv,
    sizes["year4_selected"]
)
stage_records.append(summarize_stage("year4_candidates", year4_candidates))
stage_records.append(summarize_stage("year4_selected", year4_selected))

gen4_progeny = randCross(
    year4_selected,
    nCrosses=sizes["gen4_progeny"],
    nProgeny=1,
    simParam=SP
)
stage_records.append(summarize_stage("gen4_progeny", gen4_progeny))

setPheno(year4_candidates, h2=0.35, simParam=SP)
tc2_mean = float(np.mean(year4_candidates.pheno))
tc2_locations = 5

# -------------------------
# Year 5: advanced testing and release decision
# -------------------------
year5_candidates = gen4_progeny
setPheno(year5_candidates, h2=h2, simParam=SP)
year5_ebv = add_noisy_ebv(year5_candidates, accuracy=0.75)
year5_selected, year5_idx = select_top_by_scores(
    year5_candidates,
    year5_ebv,
    sizes["year5_selected"]
)
stage_records.append(summarize_stage("year5_candidates", year5_candidates))
stage_records.append(summarize_stage("year5_selected", year5_selected))

gen5_progeny = randCross(
    year5_selected,
    nCrosses=sizes["gen5_progeny"],
    nProgeny=1,
    simParam=SP
)
stage_records.append(summarize_stage("gen5_progeny", gen5_progeny))

setPheno(year5_candidates, h2=0.50, simParam=SP)
tc3_mean = float(np.mean(year5_candidates.pheno))
tc3_locations = 20

released_variety = selectInd(
    year5_selected,
    nInd=1,
    use="gv",
    trait=0,
    selectTop=True
)
stage_records.append(summarize_stage("released_variety", released_variety))

print("Pipeline completed.")
print("TC1 mean phenotype across", tc1_locations, "locations:", tc1_mean)
print("TC2 mean phenotype across", tc2_locations, "locations:", tc2_mean)
print("TC3 mean phenotype across", tc3_locations, "locations:", tc3_mean)
print("Released variety mean genetic value:", meanG(released_variety))

## Summarize Results

In [ ]:
results_df = pd.DataFrame(stage_records)
print(results_df)

plt.figure(figsize=(10, 5))
plt.plot(results_df["stage"], results_df["meanG"], marker="o")
plt.xticks(rotation=60, ha="right")
plt.ylabel("Mean Genetic Value")
plt.title("Genetic Trend Across BRAID-Mapped Stages")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(results_df["stage"], results_df["nInd"])
plt.xticks(rotation=60, ha="right")
plt.ylabel("Number of Individuals")
plt.title("Population Size by Stage")
plt.tight_layout()
plt.show()

## Interpretation

This AlphaSimPy notebook implements the BRAID program as a runnable tutorial-style simulation.

Key features preserved from the BRAID abstraction:
- five-year stage progression,
- founder/current population initialization,
- biparental crossing,
- Year 1 phenotypic selection,
- later genomic/EBV-oriented selection,
- test-cross style evaluation checkpoints,
- and a final release decision.

If desired, this notebook can be extended with:
- explicit marker-based genomic prediction,
- separate tester populations for true test-cross simulation,
- replicated multi-environment trials,
- or multiple stochastic replicates.
